In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [25]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
# project_list = ['bids-apps_mrtrix3_connectome', 'leakec_tfc', 'voutcn_megahit', 'moble_quaternion', 'santandermetgroup_downscaler', 'rajeshrinet_pyross', 'google_jax-md', 'ncar_wrf-python', 'juliaintervals_intervalarithmetic.jl', 'magazino_move_base_flex']
projects = ['leakec_tfc']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00564d7b2c5096020d84c4433c6daf4877c86f78,leakec_tfc


In [26]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  0%|          | 0/77 [00:00<?, ?it/s]

100%|██████████| 77/77 [01:19<00:00,  1.03s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00564d7b2c5096020d84c4433c6daf4877c86f78,1a8333d9532ad3b4d5c09e92d4dd16b452bd876a,[91d4ab960ed076eb89c8490ec991bba9db2f034c],hunterjohnston <hunter.r.johnston@gmail.com>,1616528729,-0500,hunterjohnston <hunter.r.johnston@gmail.com>,1616528729,-0500,Checking to see if symlink works with example_...,00564d7b2c5096020d84c4433c6daf4877c86f78,leakec_tfc
1,00594a3931574710d96441ef713943544b720658,bb40ab0f19d675b1620a71c9b48bd2f9e8a3e247,[7e21face8326029ad9c88b943eab430ab1f5d4de],leake <carl.leake@jpl.nasa.gov>,1685288697,-0700,leake <carl.leake@jpl.nasa.gov>,1685288697,-0700,Fixing typo.\n,00594a3931574710d96441ef713943544b720658,leakec_tfc


In [27]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00564d7b2c5096020d84c4433c6daf4877c86f78', 'tree': '1a8333d9532ad3b4d5c09e92d4dd16b452bd876a', 'parent': ['91d4ab960ed076eb89c8490ec991bba9db2f034c'], 'author': 'hunterjohnston <hunter.r.johnston@gmail.com>', 'author_time': 1616528729, 'author_tz': '-0500', 'committer': 'hunterjohnston <hunter.r.johnston@gmail.com>', 'committer_time': 1616528729, 'committer_tz': '-0500', 'message': "Checking to see if symlink works with example_1_16. Looking at the 'Changes to be committed:' it seems to be working\n"}


In [28]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [29]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,leakec_tfc,00564d7b2c5096020d84c4433c6daf4877c86f78,hunterjohnston <hunter.r.johnston@gmail.com>,1616528729,Checking to see if symlink works with example_...


In [30]:
#mode a means append, so you have all your projects in the same file
yournetid='cliddel2'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github